# LSTM forecasting of conductivity at Indusii

This notebook contains only the Long Short-Term Memory (LSTM) experiment extracted from lag_analysis_fuseki.ipynb. It reads the five conductivity series from Apache Jena Fuseki and predicts the complete next four hours at the Indusii industrial site.

Important design choices:

1. The current repository Fuseki reader performs the same unit normalization used by the analytics frontend, including repair of legacy Water-Link and Waterinfo values.
2. Training, validation, and testing are separated chronologically: January-July, August, and September 2025.
3. Input and target scalers are fitted using training data only.
4. The LSTM receives unshifted 72-hour histories from all five sensors. No 42-hour delay or cross-correlation lag is manually imposed.
5. The network predicts changes from the latest Indusii value. A zero change is therefore the persistence forecast.
6. Performance is reported in mS/cm and compared with persistence at every 15-minute forecast step.
7. A completed model is saved and loaded automatically on later runs.

The earlier reconstruction-autoencoder stage has intentionally been removed. Reconstructing historical inputs is not the same objective as forecasting future conductivity, and it made the experiment slower and harder to interpret. This notebook trains a compact LSTM directly on the forecasting objective.

## 1. Imports and experiment settings

The default experiment uses 72 hours of history and produces 16 predictions: one every 15 minutes through the next four hours. Training windows are sampled once per hour to reduce near-duplicate examples and make CPU training more practical. Validation and test windows remain at full 15-minute resolution.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model, layers

# Locate the repository whether Jupyter started in the repository root or
# directly inside time_series_analysis.
_working_directory = Path.cwd().resolve()
_root_candidates = [_working_directory, _working_directory.parent]
REPO_ROOT = next(
    (path for path in _root_candidates if (path / 'lag_analytics_workspace').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        'Could not locate the repository root. Start Jupyter from the repository '
        'root or from its time_series_analysis folder.'
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from lag_analytics_workspace.fuseki import (
    CANONICAL_UNIT_URI,
    DEFAULT_GRAPH_URI,
    FusekiClient,
)

RESAMPLE_MINUTES = 15
STEPS_PER_HOUR = 60 // RESAMPLE_MINUTES
LOOKBACK_HOURS = 72
FORECAST_HOURS = 4
LOOKBACK_STEPS = LOOKBACK_HOURS * STEPS_PER_HOUR
FORECAST_STEPS = FORECAST_HOURS * STEPS_PER_HOUR

# Use one training example per hour instead of sixteen highly overlapping
# examples per four-hour interval. Validation and test still use every row.
TRAIN_STRIDE_STEPS = STEPS_PER_HOUR
BATCH_SIZE = 128
MAX_EPOCHS = 60

TRAIN_PERIOD = ('2025-01-01', '2025-07-31 23:59:59')
VALIDATION_PERIOD = ('2025-08-01', '2025-08-31 23:59:59')
TEST_PERIOD = ('2025-09-01', '2025-09-30 23:59:59')

MODEL_DIRECTORY = REPO_ROOT / 'time_series_analysis' / 'saved_models'
MODEL_NAME = f'simple_salinity_lstm_{LOOKBACK_HOURS}h_input_{FORECAST_HOURS}h_sequence_v1'
MODEL_PATH = MODEL_DIRECTORY / f'{MODEL_NAME}.keras'
HISTORY_PATH = MODEL_DIRECTORY / f'{MODEL_NAME}_history.json'
METADATA_PATH = MODEL_DIRECTORY / f'{MODEL_NAME}_metadata.json'
FORCE_RETRAIN = False

tf.keras.utils.set_random_seed(42)
print(f'Repository: {REPO_ROOT}')
print(f'Input shape: {LOOKBACK_STEPS} steps x 5 sensors ({LOOKBACK_HOURS} hours)')
print(f'Output: {FORECAST_STEPS} future changes ({FORECAST_HOURS} hours)')
print(f'TensorFlow: {tf.__version__}')

## 2. Read and normalize observations from Fuseki

The sensor identifiers below follow the current repository mapping. In particular, 289441042 is Terneuzen and 289429042 is the farther Ghent sensor. The Fuseki client reads QUDT units and returns one canonical mS/cm value per observation.

In [ ]:
GRAPH_URI = os.getenv('ANALYTICS_DEFAULT_GRAPH_URI', DEFAULT_GRAPH_URI)

SENSORS = {
    'Terneuzen': 'http://example.com/waterinfo/289441042',
    'Westdorpe': 'http://example.com/waterinfo/289435042',
    'Gent - far': 'http://example.com/waterinfo/289429042',
    'Gent - near': 'http://example.com/waterinfo/289423042',
    'Indusii': 'http://example.com/waterlink/111111111',
}
TARGET = 'Indusii'
SENSOR_NAMES = list(SENSORS)

client = FusekiClient(timeout=120)
print(f'Fuseki query endpoint: {client.endpoint}')
print(f'Named graph: {GRAPH_URI}')

raw = client.observations(
    GRAPH_URI,
    SENSORS.values(),
    limit=250_000,
    cache_seconds=0,
)
raw['time'] = pd.to_datetime(raw['time'], utc=True)
raw['result'] = pd.to_numeric(raw['result'], errors='coerce')

wide = raw.pivot_table(
    index='time', columns='sensor', values='result', aggfunc='mean'
).rename(columns={uri: name for name, uri in SENSORS.items()}).sort_index()

missing_sensors = [name for name in SENSOR_NAMES if name not in wide.columns]
if missing_sensors:
    raise RuntimeError(
        f'Fuseki did not return the required sensors: {missing_sensors}. '
        f'Available columns: {list(wide.columns)}'
    )

unit_report = pd.DataFrame(raw.attrs.get('unit_report', []))
if not unit_report.empty:
    unit_report['sensor_name'] = unit_report['sensor'].map(
        {uri: name for name, uri in SENSORS.items()}
    )
    display(unit_report[['sensor_name', 'source_units', 'methods', 'legacy_graph_repair']])

print(f'Retrieved {len(raw):,} normalized observations in {CANONICAL_UNIT_URI}.')
display(wide[SENSOR_NAMES].head())

## 3. Build a leakage-safe 15-minute table

Only the common observed interval of all five sensors is retained. Internal gaps of at most one hour are interpolated. Longer gaps remain missing, and any window that crosses one is rejected. A trailing two-hour mean suppresses isolated measurement noise while using only present and past observations.

In [ ]:
data = wide[SENSOR_NAMES].apply(pd.to_numeric, errors='coerce')
first_observation = data.apply(pd.Series.first_valid_index)
last_observation = data.apply(pd.Series.last_valid_index)
common_start = first_observation.max()
common_end = last_observation.min()

if pd.isna(common_start) or pd.isna(common_end) or common_start >= common_end:
    raise RuntimeError('The five sensors do not have a valid common observation period.')

data = data.loc[common_start:common_end].resample(f'{RESAMPLE_MINUTES}min').mean()
data = data.interpolate(method='time', limit=4, limit_area='inside')
data = data.rolling(
    window=2 * STEPS_PER_HOUR,
    min_periods=2 * STEPS_PER_HOUR,
).mean()

coverage = pd.DataFrame({
    'first observation': first_observation,
    'last observation': last_observation,
    'missing rows after preparation': data.isna().sum(),
    'minimum mS/cm': data.min(),
    'maximum mS/cm': data.max(),
})
display(coverage)
print(f'Common period: {common_start} to {common_end}')

data.plot(subplots=True, figsize=(14, 10), sharex=True)
plt.suptitle('Prepared conductivity series used by the LSTM (mS/cm)', y=1.01)
plt.tight_layout()
plt.show()

## 4. Scale the data and create chronological windows

The input scaler learns one mean and standard deviation per sensor from complete January-July rows only. Each target is the future Indusii value minus the latest known Indusii value. Target changes are scaled separately for each forecast horizon because uncertainty normally grows with lead time.

Windows are assigned to a split by their forecast times. August and September inputs may therefore include the immediately preceding history, but their target values never enter training.

In [ ]:
training_rows = data.loc[TRAIN_PERIOD[0]:TRAIN_PERIOD[1], SENSOR_NAMES].dropna()
if training_rows.empty:
    raise RuntimeError('No complete training rows were found in the configured period.')

input_scaler = StandardScaler().fit(training_rows)
scaled_values = input_scaler.transform(data[SENSOR_NAMES]).astype('float32')
target_values = data[TARGET].to_numpy(dtype='float32')
timestamps = data.index

def make_windows(period, stride_steps=1):
    """Create complete histories and future target-change sequences."""
    period_start = pd.Timestamp(period[0], tz='UTC')
    period_end = pd.Timestamp(period[1], tz='UTC')
    X, y, current, forecast_times = [], [], [], []

    # s is the first forecast step; the latest input is s - 1.
    for s in range(LOOKBACK_STEPS, len(data) - FORECAST_STEPS + 1, stride_steps):
        future_indices = slice(s, s + FORECAST_STEPS)
        first_forecast_time = timestamps[s]
        final_forecast_time = timestamps[s + FORECAST_STEPS - 1]
        if first_forecast_time < period_start or final_forecast_time > period_end:
            continue

        history = scaled_values[s - LOOKBACK_STEPS:s]
        latest_target = target_values[s - 1]
        future_targets = target_values[future_indices]
        if not (
            np.isfinite(history).all()
            and np.isfinite(latest_target)
            and np.isfinite(future_targets).all()
        ):
            continue

        X.append(history)
        y.append(future_targets - latest_target)
        current.append(latest_target)
        forecast_times.append(final_forecast_time)

    return (
        np.asarray(X, dtype='float32'),
        np.asarray(y, dtype='float32'),
        np.asarray(current, dtype='float32'),
        pd.DatetimeIndex(forecast_times),
    )

X_train, y_train, current_train, time_train = make_windows(
    TRAIN_PERIOD, TRAIN_STRIDE_STEPS
)
X_val, y_val, current_val, time_val = make_windows(VALIDATION_PERIOD)
X_test, y_test, current_test, time_test = make_windows(TEST_PERIOD)

for split_name, values in {
    'training': X_train,
    'validation': X_val,
    'testing': X_test,
}.items():
    if len(values) == 0:
        raise RuntimeError(f'No valid {split_name} windows were created.')

target_scale = y_train.std(axis=0)
target_scale = np.where(target_scale < 1e-6, 1.0, target_scale).astype('float32')
y_train_scaled = y_train / target_scale
y_val_scaled = y_val / target_scale

print('X shape: examples, historical time steps, sensors')
print('y shape: examples, future 15-minute steps')
print('Training:  ', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Testing:   ', X_test.shape, y_test.shape)

## 5. Build the LSTM forecaster

An LSTM reads the sensor history in chronological order. Its gates decide what information to retain, update, or forget as it moves through the 288 input steps. The first LSTM returns an encoded sequence; the second condenses it into one state used by a small dense forecasting head.

The final layer starts with zero weights. Before learning, all predicted changes are zero, exactly reproducing persistence. Huber loss is optimized because it is less dominated by occasional extreme salinity errors than mean-squared error. Mean absolute error is recorded as an additional training metric.

In [ ]:
def build_lstm():
    inputs = layers.Input(
        shape=(LOOKBACK_STEPS, len(SENSOR_NAMES)), name='sensor_history'
    )
    x = layers.LSTM(64, return_sequences=True, name='lstm_history')(inputs)
    x = layers.Dropout(0.10)(x)
    x = layers.LSTM(32, return_sequences=False, name='lstm_summary')(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dense(32, activation='relu', name='forecast_features')(x)
    x = layers.Dropout(0.10)(x)
    outputs = layers.Dense(
        FORECAST_STEPS,
        kernel_initializer='zeros',
        bias_initializer='zeros',
        name='scaled_future_changes',
    )(x)
    return Model(inputs, outputs, name='simple_salinity_lstm')

def compile_lstm(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=[tf.keras.metrics.MeanAbsoluteError(name='scaled_mae')],
    )
    return model

untrained_model = compile_lstm(build_lstm())
untrained_model.summary()

## 6. Load a completed model, or train and save a new one

Training is skipped only when both the Keras model and its completion metadata exist. During training, the best August validation checkpoint is written to disk. The metadata file is created only after model.fit finishes, so an interrupted run is not mistaken for a completed experiment.

Set FORCE_RETRAIN to True in section 1 when you deliberately want to replace the saved model.

In [ ]:
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

expected_metadata = {
    'model_name': MODEL_NAME,
    'sensor_names': SENSOR_NAMES,
    'lookback_hours': LOOKBACK_HOURS,
    'forecast_hours': FORECAST_HOURS,
    'resample_minutes': RESAMPLE_MINUTES,
}

can_load = MODEL_PATH.exists() and METADATA_PATH.exists() and not FORCE_RETRAIN
if can_load:
    saved_metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
    for key, expected_value in expected_metadata.items():
        if saved_metadata.get(key) != expected_value:
            raise RuntimeError(
                f'Saved-model metadata mismatch for {key}: '
                f'{saved_metadata.get(key)!r} != {expected_value!r}. '
                'Set FORCE_RETRAIN = True.'
            )

    # A model is valid only with the preprocessing values used during training.
    scaler_checks = {
        'input_scaler_mean': input_scaler.mean_,
        'input_scaler_scale': input_scaler.scale_,
        'target_change_scale_by_horizon': target_scale,
    }
    for key, current_values in scaler_checks.items():
        saved_values = np.asarray(saved_metadata.get(key, []), dtype='float64')
        if saved_values.shape != current_values.shape or not np.allclose(
            saved_values, current_values, rtol=1e-5, atol=1e-7
        ):
            raise RuntimeError(
                f'The current Fuseki training data produce different {key}. '
                'Set FORCE_RETRAIN = True so the model and preprocessing stay aligned.'
            )

    print(f'Loading completed model: {MODEL_PATH}')
    model = tf.keras.models.load_model(MODEL_PATH)
    history_values = (
        json.loads(HISTORY_PATH.read_text(encoding='utf-8'))
        if HISTORY_PATH.exists()
        else None
    )
else:
    print('Training a new LSTM forecaster...')
    model = untrained_model
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            MODEL_PATH,
            monitor='val_loss',
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            patience=4,
            factor=0.5,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        y_train_scaled,
        validation_data=(X_val, y_val_scaled),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        shuffle=False,
        verbose=1,
    )
    history_values = {
        name: [float(value) for value in values]
        for name, values in history.history.items()
    }
    HISTORY_PATH.write_text(json.dumps(history_values, indent=2), encoding='utf-8')

    completed_metadata = {
        **expected_metadata,
        'train_period': list(TRAIN_PERIOD),
        'validation_period': list(VALIDATION_PERIOD),
        'test_period': list(TEST_PERIOD),
        'graph_uri': GRAPH_URI,
        'fuseki_endpoint': client.endpoint,
        'canonical_unit': CANONICAL_UNIT_URI,
        'input_scaler_mean': input_scaler.mean_.tolist(),
        'input_scaler_scale': input_scaler.scale_.tolist(),
        'target_change_scale_by_horizon': target_scale.tolist(),
    }
    METADATA_PATH.write_text(
        json.dumps(completed_metadata, indent=2), encoding='utf-8'
    )
    # Reload the best checkpoint rather than the final epoch.
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f'Saved best model: {MODEL_PATH}')
    print(f'Saved history: {HISTORY_PATH}')
    print(f'Saved preprocessing metadata: {METADATA_PATH}')

model.summary()

### 6.1 Plot the training history

In [ ]:
def plot_training_history(saved_history):
    if saved_history is None:
        print('No saved training history was found; model evaluation can still run.')
        return

    train_loss = np.asarray(saved_history['loss'])
    validation_loss = np.asarray(saved_history['val_loss'])
    epochs_axis = np.arange(1, len(train_loss) + 1)
    best_epoch = int(np.argmin(validation_loss)) + 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].plot(epochs_axis, train_loss, label='Training Huber loss')
    axes[0].plot(epochs_axis, validation_loss, label='Validation Huber loss')
    axes[0].axvline(best_epoch, color='black', linestyle='--', alpha=0.6)
    axes[0].set_title(f'Huber loss (best validation epoch: {best_epoch})')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Scaled Huber loss')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    if 'scaled_mae' in saved_history and 'val_scaled_mae' in saved_history:
        axes[1].plot(epochs_axis, saved_history['scaled_mae'], label='Training MAE')
        axes[1].plot(epochs_axis, saved_history['val_scaled_mae'], label='Validation MAE')
        axes[1].set_title('Additional MAE metric during training')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Scaled MAE')
        axes[1].grid(alpha=0.3)
        axes[1].legend()
    else:
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

plot_training_history(history_values)

### 6.2 Verify the saved files

In [ ]:
for label, path in {
    'Model': MODEL_PATH,
    'History': HISTORY_PATH,
    'Metadata': METADATA_PATH,
}.items():
    print(f'{label:8}: exists={path.exists()} | {path.resolve()}')

## 7. Evaluate the LSTM against persistence

Persistence repeats the latest observed Indusii conductivity at every future step. All metrics below are calculated after converting the LSTM output back to mS/cm. Positive RMSE skill means the LSTM improves on persistence; zero means equal performance; a negative value means persistence is better.

In [ ]:
predicted_change_scaled = model.predict(X_test, batch_size=256, verbose=1)
predicted_change = predicted_change_scaled * target_scale[None, :]

actual_future = current_test[:, None] + y_test
lstm_prediction = current_test[:, None] + predicted_change
persistence_prediction = np.repeat(
    current_test[:, None], FORECAST_STEPS, axis=1
)

def regression_scores(actual, prediction):
    return {
        'MAE (mS/cm)': mean_absolute_error(actual, prediction),
        'RMSE (mS/cm)': np.sqrt(mean_squared_error(actual, prediction)),
        'R2': r2_score(actual, prediction),
    }

overall_rows = []
for model_name, prediction in {
    'LSTM': lstm_prediction,
    'Persistence': persistence_prediction,
}.items():
    overall_rows.append({
        'Model': model_name,
        **regression_scores(actual_future.ravel(), prediction.ravel()),
    })

overall_metrics = pd.DataFrame(overall_rows)
display(overall_metrics.round(4))

lstm_rmse = overall_metrics.loc[
    overall_metrics['Model'] == 'LSTM', 'RMSE (mS/cm)'
].iloc[0]
persistence_rmse = overall_metrics.loc[
    overall_metrics['Model'] == 'Persistence', 'RMSE (mS/cm)'
].iloc[0]
overall_skill = 1.0 - lstm_rmse / persistence_rmse
print(f'Overall RMSE skill relative to persistence: {overall_skill:.2%}')
if overall_skill <= 0:
    print('The LSTM did not beat persistence overall in this test period.')

### 7.1 Metrics at every forecast horizon

In [ ]:
horizon_rows = []
for step in range(FORECAST_STEPS):
    lead_minutes = (step + 1) * RESAMPLE_MINUTES
    actual_step = actual_future[:, step]
    lstm_step = lstm_prediction[:, step]
    persistence_step = persistence_prediction[:, step]

    lstm_scores = regression_scores(actual_step, lstm_step)
    persistence_scores = regression_scores(actual_step, persistence_step)
    skill = 1.0 - (
        lstm_scores['RMSE (mS/cm)'] / persistence_scores['RMSE (mS/cm)']
    )
    horizon_rows.append({
        'Lead minutes': lead_minutes,
        'LSTM MAE': lstm_scores['MAE (mS/cm)'],
        'Persistence MAE': persistence_scores['MAE (mS/cm)'],
        'LSTM RMSE': lstm_scores['RMSE (mS/cm)'],
        'Persistence RMSE': persistence_scores['RMSE (mS/cm)'],
        'LSTM R2': lstm_scores['R2'],
        'Persistence R2': persistence_scores['R2'],
        'RMSE skill': skill,
    })

horizon_metrics = pd.DataFrame(horizon_rows)
hourly_summary = horizon_metrics[
    horizon_metrics['Lead minutes'].isin([60, 120, 180, 240])
]
display(hourly_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(
    horizon_metrics['Lead minutes'] / 60,
    horizon_metrics['LSTM RMSE'],
    marker='o',
    label='LSTM',
)
axes[0].plot(
    horizon_metrics['Lead minutes'] / 60,
    horizon_metrics['Persistence RMSE'],
    marker='o',
    label='Persistence',
)
axes[0].set_xlabel('Forecast lead time (hours)')
axes[0].set_ylabel('RMSE (mS/cm)')
axes[0].set_title('Error by forecast horizon')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(
    horizon_metrics['Lead minutes'] / 60,
    100 * horizon_metrics['RMSE skill'],
    marker='o',
)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Forecast lead time (hours)')
axes[1].set_ylabel('RMSE skill over persistence (%)')
axes[1].set_title('Positive values indicate improvement')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 7.2 Inspect the four-hour endpoint

The final output column corresponds to the same four-hour endpoint used in the simple TCN notebook.

In [ ]:
endpoint_actual = actual_future[:, -1]
endpoint_lstm = lstm_prediction[:, -1]
endpoint_persistence = persistence_prediction[:, -1]

endpoint_metrics = pd.DataFrame([
    {'Model': 'LSTM', **regression_scores(endpoint_actual, endpoint_lstm)},
    {
        'Model': 'Persistence',
        **regression_scores(endpoint_actual, endpoint_persistence),
    },
])
display(endpoint_metrics.round(4))

endpoint_predictions = pd.DataFrame({
    'Actual': endpoint_actual,
    'LSTM': endpoint_lstm,
    'Persistence': endpoint_persistence,
}, index=time_test)

week = endpoint_predictions.loc['2025-09-05':'2025-09-12']
week.plot(figsize=(15, 5), linewidth=1.4)
plt.title('Indusii conductivity: four-hour-ahead LSTM forecast')
plt.xlabel('Forecast time')
plt.ylabel('Conductivity (mS/cm)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Produce a forecast from the latest complete history

This final cell is operational rather than a test: it forecasts beyond the last complete input row available from Fuseki. These future values do not yet have observations against which metrics can be calculated.

In [ ]:
complete_history_ends = []
for end in range(LOOKBACK_STEPS, len(data) + 1):
    history = scaled_values[end - LOOKBACK_STEPS:end]
    current_value = target_values[end - 1]
    if np.isfinite(history).all() and np.isfinite(current_value):
        complete_history_ends.append(end)

if not complete_history_ends:
    raise RuntimeError('No complete latest history is available for forecasting.')

latest_end = complete_history_ends[-1]
latest_input = scaled_values[
    latest_end - LOOKBACK_STEPS:latest_end
][None, ...]
latest_current = float(target_values[latest_end - 1])
latest_time = timestamps[latest_end - 1]

latest_scaled_change = model.predict(latest_input, verbose=0)[0]
latest_forecast = latest_current + latest_scaled_change * target_scale
future_times = pd.date_range(
    latest_time + pd.Timedelta(minutes=RESAMPLE_MINUTES),
    periods=FORECAST_STEPS,
    freq=f'{RESAMPLE_MINUTES}min',
)
latest_forecast_table = pd.DataFrame({
    'Forecast Indusii conductivity (mS/cm)': latest_forecast,
    'Persistence (mS/cm)': latest_current,
}, index=future_times)

print(f'Latest observed input time: {latest_time}')
display(latest_forecast_table)
latest_forecast_table.plot(figsize=(12, 4), marker='o')
plt.title('Latest available four-hour Indusii forecast')
plt.xlabel('Forecast time')
plt.ylabel('Conductivity (mS/cm)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. How to interpret the result

- The LSTM has access to 72 hours of unshifted observations but is not told that Terneuzen should lead Indusii by 42 hours. Any useful delay must be learned from the sequences.
- The 72-hour input is a maximum available context, not a claim that every hour is useful.
- Positive test skill over persistence is required before claiming an improvement.
- Report the four-hour endpoint when comparing with the simple TCN, and use the horizon table when discussing the complete forecast trajectory.
- September remains untouched until final evaluation; do not tune architecture choices using the September scores repeatedly.
- For a stronger conference comparison, repeat the experiment with several chronological test months and compare the five-sensor model with an Indusii-only ablation.